In [2]:
import pandas as pd
df=pd.read_csv("600_gabarito.csv")
df=df.iloc[:10]
df.to_csv('just_test.csv',index=False)

In [2]:
# -*- coding: utf-8 -*-
"""
avaliar_pipeline_hibrido_final_robusto.py

Este script é uma fusão do 'avaliar_pipeline.py' (lógica híbrida)
com o 'avaliar_pipeline_structured.py' (robustez).

CARACTERÍSTICAS:
1.  EXTRAÇÃO HÍBRIDA:
    - municipio: Solr
    - edital: Regex
    - modalidade: LLM
    - objeto: LLM
2.  MÉTRICAS DUPLAS: Calcula "Restritiva" (acerto exato) e "Contém" (menos restritiva).
3.  AVALIAÇÃO MANUAL: Carrega métricas manuais (ex: 'validaçao_objeto') 
    para atributos desabilitados no 'AVALIAR'.
4.  PROCESSAMENTO ROBUSTO:
    - Processa em lotes (START_INDEX / END_INDEX).
    - Salva parcialmente (SAVE_EVERY).
    - Pode resumir (RESUME_PROCESSING) de onde parou.
5.  LÓGICA CORRIGIDA: Inclui todas as correções de bug (case-insensitive, 
   múltiplos valores, [[]], e lógica FN/FP estrita).
6.  OUTPUT SIMPLIFICADO: Nível 2 foca apenas na Média das Acurácias.
"""

import pandas as pd
import re
import pysolr  # <-- Mantido para lógica híbrida
import os
import json
import time
from tqdm import tqdm
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from typing import List, Dict, Optional, Any, Tuple

# ==============================================================================
# TODO 1: CONFIGURAÇÕES PRINCIPAIS 
# ==============================================================================
CAMINHO_CSV = "196_validation.csv"  # <-- Seu arquivo principal de gabarito

# ==============================================================================
# TODO 1.5: MAPEAMENTO DAS COLUNAS
# ==============================================================================
COLUNAS_GT = {
    'texto_noticia': 'texto_noticia', # Coluna com o texto
    
    'municipio': {
        'valor': 'municipio_ente',         
        'presente': 'municipio_presente'  
    },
    'modalidade': {
        'valor': 'modalidade_licitacao',   
        'presente': 'modalidade_presente'  
    },
    'edital': {
        'valor': 'edital',                 
        'presente': 'edital_presente'      
    },
    'objeto': {
        'valor': 'objeto',                 
        'presente': 'objeto_presente'      
    }
}

# ==============================================================================
# TODO 1.6: HABILITAR/DESABILITAR ATRIBUTOS
# ==============================================================================
AVALIAR = {
    'municipio': True,   # <-- True = Rodar Solr
    'modalidade': True,  # <-- True = Rodar LLM
    'edital': True,      # <-- True = Rodar Regex
    'objeto': False     # <-- False = Carregar Métrica Manual
}

# ==============================================================================
# TODO 1.7: MAPEAMENTO DE COLUNAS DE VALIDAÇÃO MANUAL
# ==============================================================================
COLUNAS_MANUAIS = {
    'objeto': 'validaçao_objeto' 
}

# ==============================================================================
# TODO 2: CONFIGURAÇÕES DAS FERRAMENTAS
# ==============================================================================
SOLR_URL_BASE = "http://localhost:8983/solr" # <-- Mantido
OLLAMA_HOST = "https://ollama-dev.ceos.ufsc.br"
SELECTED_MODEL = "gpt-oss:20b"
LLM_TEMPERATURE = 0

# =================================
# TODO 3: PARÂMETROS DE EXECUÇÃO
# =================================
START_INDEX = 0         # Linha inicial (baseado em 0)
END_INDEX = -1          # Linha final (-1 para processar até o fim)
SAVE_EVERY = 20         # Salvar um backup a cada X iterações
RESUME_PROCESSING = True # Pular linhas que já foram processadas
CAMINHO_CSV_PARCIAL = "analise_parcial_hibrido.csv"
CAMINHO_CSV_FINAL = "analise_geral_hibrido.csv"


# ==============================================================================
# FUNÇÕES DE EXTRAÇÃO (HÍBRIDAS)
# ==============================================================================

def encontrar_municipios(texto: str, solr_conn: pysolr.Solr) -> List[str]:
    texto_lower = texto.lower()
    municipios_encontrados = []
    try:
        # Tenta buscar do cache do Solr
        for doc in solr_conn.search('*:*', rows=300):
            municipio_lower = doc.get('municipio_lower', [''])[0]
            municipio = doc.get('municipio', [''])[0]
            if re.search(rf"\b{re.escape(municipio_lower)}\b", texto_lower):
                municipios_encontrados.append(municipio)
    except Exception as e:
        tqdm.write(f"[Solr Error: municipio] {e}")
    return list(set(municipios_encontrados))

def encontrar_editais(texto: str) -> List[str]:
    editais_encontrados = []
    pattern = r'\b(?:n\.?|nº)?\s*([A-Z]{0,3}\d+/20[12]\d)\b'
    matches = re.finditer(pattern, texto, re.IGNORECASE)
    for match in matches:
        edital = match.group(1)
        if edital not in editais_encontrados:
            editais_encontrados.append(edital)
    return editais_encontrados

def extract_info_llm(text: str, llm: ChatOllama) -> (List[str], List[str]):
    """
    Extrai 'objeto' E 'modalidade' usando LLM.
    Retorna (objeto_pred, modalidade_pred)
    """
    prompt = f"""
Você é um assistente especializado em análise de notícias sobre licitações públicas.
Sua tarefa é ler atentamente o texto de uma notícia e identificar os seguintes atributos, quando estiverem explicitamente presentes:

1.  **objeto** — descreve o que está sendo licitado (ex: "construção de obra", "prestação de serviços").
    *Retorne exatamente o trecho da notícia que identifica o objeto.*

2) modalidade: A modalidade de licitação mencionada na notícia.
   - Exemplos: pregão presencial, pregão eletrônico, concorrência pública, 
     tomada de preços, dispensa de licitação, 
     inexigibilidade de licitação, leilão, concurso, regime diferenciado de contratação
   
   **REGRAS DE NORMALIZAÇÃO E EXCLUSÃO (MUITO IMPORTANTE):**
   
   - **NORMALIZAR PLURAL:** Converta plurais para singular (ex: "pregões" -> "pregão", "dispensas" -> "dispensa").
   - **NORMALIZAR TERMOS:**
     - Por exemplo, se o texto citar "dispensa", "inexibilidade" (no singular ou plural), normalize para "dispensa de licitação", inexigibilidade de licitação, etc..
     - Se for CERTEZA que "concorrência" é uma modalidade, normalize para "concorrência pública".

   - **O QUE IGNORAR (NÃO SÃO MODALIDADES):**
     - **NÃO EXTRAIA:** "registro de preço" ou "sistema de registro de preço" (isto é um procedimento, não uma modalidade).
     - **NÃO EXTRAIA:** "pregão público" (é um termo genérico, não uma modalidade).
     - **IGNORAR "concorrência":** Não extraia a palavra "concorrência" quando usada no sentido de competição (ex: "frustrar a concorrência").
     - **IGNORAR "concurso":** Não extraia "concurso" quando for seleção de pessoal (ex: "concurso público para cargos").

Retorne o resultado exclusivamente no formato JSON a seguir:
{{
  "objeto": "texto extraído ou []",
  "modalidade": "texto extraído ou []" 
}}

Texto da notícia:
\"\"\"{text}\"\"\"
"""
    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        result = response.content.strip()
        
        try:
            if result.startswith("```json"): result = result[7:]
            if result.endswith("```"): result = result[:-3]
            result = result.strip()
            data = json.loads(result)
        except json.JSONDecodeError:
            try:
                data = eval(result.replace("null", "None")) if result.startswith("{") else {"objeto": "[]", "modalidade": "[]"}
            except Exception:
                 data = {"objeto": "[]", "modalidade": "[]"}
        except Exception as e:
            tqdm.write(f"[LLM JSON Parse Error] {e}\nResponse: {result}")
            data = {"objeto": "[]", "modalidade": "[]"}

        obj_str = data.get("objeto", "[]")
        mod_str = data.get("modalidade", "[]") 
        
        empty_vals = ["", "[]", None, "[[]]"]
        
        objeto_pred = [] if obj_str in empty_vals or (isinstance(obj_str, list) and len(obj_str) == 0) else ([obj_str] if isinstance(obj_str, str) else obj_str)
        modalidade_pred = [] if mod_str in empty_vals or (isinstance(mod_str, list) and len(mod_str) == 0) else ([mod_str] if isinstance(mod_str, str) else mod_str)
        
        objeto_pred = [str(item) for item in objeto_pred if str(item).strip()]
        modalidade_pred = [str(item) for item in modalidade_pred if str(item).strip()]

        return objeto_pred, modalidade_pred

    except Exception as e:
        # Captura erros de rede como 404
        tqdm.write(f"[LLM Invocation Error] {e}")
        return ["ERROR"], ["ERROR"]

# ==============================================================================
# FUNÇÕES DE CÁLCULO DE MÉTRICA
# ==============================================================================

def clean_gt_value(value: Any) -> Optional[str]:
    if pd.isna(value) or value is None:
        return None
    val_str = str(value).strip()
    if val_str in ["", "[]", "0", "-"]:
        return None
    return val_str

def normalize_edital(edital_str: str) -> str:
    if edital_str is None:
        return None
    parts = edital_str.split('/')
    if len(parts) == 2:
        num_norm = parts[0].lstrip('0')
        if not num_norm: num_norm = '0'
        return f"{num_norm}/{parts[1]}"
    else:
        return edital_str

def get_binary_metrics(gt_val: Optional[str], pred_list: List[str], attr: str) -> \
                       Tuple[int, int, int, int, str, int, int, int, int, str]:
    has_gt = gt_val is not None      
    has_pred = bool(pred_list)    
    is_correct_contains = False
    is_correct_strict = False
    gt_set_norm = set()
    pred_set_norm = set()

    if has_gt:
        if attr == 'edital':
            gt_set_norm = {normalize_edital(item.strip()) for item in gt_val.split(',') if item.strip()}
        else:
            gt_set_norm = {item.strip().lower() for item in gt_val.split(',') if item.strip()}
    if has_pred:
        if attr == 'edital':
             pred_set_norm = {normalize_edital(p) for p in pred_list if str(p).strip()}
        else:
             pred_set_norm = {str(p).lower() for p in pred_list if str(p).strip()}

    if has_gt and has_pred:
        is_correct_contains = gt_set_norm.issubset(pred_set_norm)
        is_correct_strict = (gt_set_norm == pred_set_norm) 

    tp_c, fp_c, fn_c, tn_c = 0, 0, 0, 0; classif_c = ""
    if has_gt and is_correct_contains: tp_c = 1; classif_c = "TP"
    elif has_gt and not is_correct_contains: fn_c = 1; classif_c = "FN"
    elif not has_gt and has_pred: fp_c = 1; classif_c = "FP"
    elif not has_gt and not has_pred: tn_c = 1; classif_c = "TN"
        
    tp_s, fp_s, fn_s, tn_s = 0, 0, 0, 0; classif_s = ""
    if has_gt and is_correct_strict: tp_s = 1; classif_s = "TP"
    elif has_gt and not has_pred: fn_s = 1; classif_s = "FN"
    elif has_gt and has_pred and not is_correct_strict: fp_s = 1; classif_s = "FP"
    elif not has_gt and has_pred: fp_s = 1; classif_s = "FP"
    elif not has_gt and not has_pred: tn_s = 1; classif_s = "TN"
        
    return tp_c, fp_c, fn_c, tn_c, classif_c, tp_s, fp_s, fn_s, tn_s, classif_s

def calculate_prf1(tp: int, fp: int, fn: int, tn: int) -> Dict[str, float]:
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
    return {
        'accuracy': accuracy, 'precision': precision, 'recall': recall,
        'f1_score': f1, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn
    }

def print_metrics_table(tabela: Dict, metric_type: str):
    print("-------------------------------------------------------------------------------------")
    print(f"| Atributo          | Acurácia  | Precisão  | Recall    | F1-Score  | TP   | FP   | FN   | TN   |")
    print("-------------------------------------------------------------------------------------")
    for attr, metrics_by_type in tabela.items():
        m = metrics_by_type.get(metric_type) 
        if not m: print(f"| {attr:<17} | {'METRIC MISSING':<61} |")
        elif 'status' in m and m['status'] not in ['MANUAL']: print(f"| {attr:<17} | {m['status']:<61} |")
        else:
            status_marker = " (Manual)" if m.get('status') == 'MANUAL' else ""
            attr_display = f"{attr}{status_marker}"
            print(f"| {attr_display:<17} | {m['accuracy']:<9.2%} | {m['precision']:<9.2%} | {m['recall']:<9.2%} | {m['f1_score']:<9.2%} | {int(m['tp']):<4} | {int(m['fp']):<4} | {int(m['fn']):<4} | {int(m['tn']):<4} |")
    print("-------------------------------------------------------------------------------------")

# ==============================================================================
# FUNÇÃO PRINCIPAL
# ==============================================================================

def run_evaluation():
    print("Iniciando pipeline de avaliação HÍBRIDO (Solr/Regex/LLM)...")

    # --- 1. Conectar às ferramentas ---
    solr_municipios = None
    if AVALIAR.get('municipio', False):
        print(f"Conectando ao Solr ({SOLR_URL_BASE})...")
        try:
            solr_municipios = pysolr.Solr(f'{SOLR_URL_BASE}/municipios', timeout=10)
            solr_municipios.search('*:*', rows=1)
            print("Conexão ao Solr ('municipios') bem-sucedida.")
        except Exception as e:
            print(f"ERRO: Não foi possível conectar ao Solr. 'municipio' não será processado.")
            AVALIAR['municipio'] = False # Desabilita se não puder conectar

    llm = None
    needs_llm = AVALIAR.get('objeto', False) or AVALIAR.get('modalidade', False)
    if needs_llm:
        print(f"Conectando ao Ollama ({OLLAMA_HOST}) com modelo {SELECTED_MODEL}...")
        try:
            os.environ["OLLAMA_HOST"] = OLLAMA_HOST
            llm = ChatOllama(model=SELECTED_MODEL, base_url=OLLAMA_HOST, temperature=LLM_TEMPERATURE)
            # O primeiro 'invoke' real testará a conexão
            print("Cliente Ollama preparado.")
        except Exception as e:
            print(f"ERRO: Não foi possível inicializar o cliente Ollama. Detalhe: {e}")
            return
    
    # --- 2. Carregar e Preparar os Dados ---
    print(f"Carregando gabarito (Ground Truth) de '{CAMINHO_CSV}'...")
    try:
        df = pd.read_csv(CAMINHO_CSV)
    except FileNotFoundError:
        print(f"ERRO: Arquivo '{CAMINHO_CSV}' não encontrado."); return
    except Exception as e:
        print(f"ERRO ao carregar CSV: {e}"); return

    # --- 3. Configurar parâmetros de execução (Start/End/Resume) ---
    total_rows = len(df)
    global_start = START_INDEX
    global_end = END_INDEX
    if global_end == -1 or global_end > total_rows: global_end = total_rows
    if global_start < 0: global_start = 0
    if global_start >= global_end:
        print(f"START_INDEX ({global_start}) é >= END_INDEX ({global_end}). Nada a processar."); return
    num_to_process = global_end - global_start
    print(f"Total de {total_rows} itens carregados. Processando fatia de {global_start} a {global_end-1} ({num_to_process} itens).")

    # --- 4. Processar Notícias (Loop Principal) ---
    attributes_habilitados = [attr for attr, enabled in AVALIAR.items() if enabled]
    if attributes_habilitados:
        print(f"Atributos a serem avaliados AUTOMATICAMENTE: {', '.join(attributes_habilitados)}")
    else:
        print("Nenhum atributo habilitado para avaliação automática. Rodando apenas métricas manuais.")

    marker_col = None
    is_resuming = RESUME_PROCESSING
    if attributes_habilitados:
        marker_col = f'classif_strict_{attributes_habilitados[0]}'
        if marker_col not in df.columns:
            print(f"Coluna marcador '{marker_col}' não encontrada. Desabilitando RESUME.")
            is_resuming = False
            # Criar colunas vazias se não existirem
            for attr in attributes_habilitados:
                df[f'gt_raw_{attr}'] = None
                df[f'gt_presente_{attr}'] = None
                df[f'gt_efetivo_{attr}'] = None
                df[f'pred_{attr}'] = None
                df[f'classif_contains_{attr}'] = None
                df[f'classif_strict_{attr}'] = None
        elif is_resuming:
            print(f"RESUME habilitado. Verificando marcador '{marker_col}' para pular linhas.")
    else:
        is_resuming = False

    for idx, row in tqdm(df.iloc[global_start:global_end].iterrows(), total=num_to_process, desc="Avaliando notícias"):
        try:
            if is_resuming and pd.notna(row.get(marker_col)):
                continue
            text = str(row[COLUNAS_GT['texto_noticia']])
        except KeyError:
            tqdm.write(f"ERRO: Coluna de texto '{COLUNAS_GT['texto_noticia']}' não encontrada."); return

        pred_municipio = []; pred_modalidade = []; pred_edital = []; pred_objeto = []

        if AVALIAR.get('municipio', False) and solr_municipios: 
            pred_municipio = encontrar_municipios(text, solr_municipios)
        if AVALIAR.get('edital', False): 
            pred_edital = encontrar_editais(text)
            
        if needs_llm and llm:
            temp_objeto, temp_modalidade = extract_info_llm(text, llm)
            if AVALIAR.get('objeto', False):
                pred_objeto = temp_objeto
            if AVALIAR.get('modalidade', False):
                pred_modalidade = temp_modalidade
        elif needs_llm and not llm:
             tqdm.write("ERRO: LLM é necessário mas não está conectado.")

        pred_dict = {'municipio': pred_municipio, 'modalidade': pred_modalidade, 'edital': pred_edital, 'objeto': pred_objeto}

        for attr in attributes_habilitados:
            try:
                col_config = COLUNAS_GT[attr]; valor_gt_bruto = row.get(col_config['valor'])
                valor_presente_bruto = row.get(col_config['presente'])
                is_present_in_text = str(valor_presente_bruto).strip() in ['1', '1.0']
            except KeyError as e:
                tqdm.write(f"\nERRO: Coluna não encontrada: {e}. Verifique `COLUNAS_GT`."); return

            gt_val_efetivo = None
            if is_present_in_text: gt_val_efetivo = clean_gt_value(valor_gt_bruto)
            
            pred_list = pred_dict.get(attr, [])
            
            # Se a predição for ["ERROR"] (por ex, erro 404), registre como FN
            if pred_list == ["ERROR"]:
                classif_c = "FN" if has_gt else "TN" # Se tinha gabarito é FN, senão TN
                classif_s = "FN" if has_gt else "TN"
                tp_c, fp_c, fn_c, tn_c = (0, 0, 1, 0) if has_gt else (0, 0, 0, 1)
                tp_s, fp_s, fn_s, tn_s = (0, 0, 1, 0) if has_gt else (0, 0, 0, 1)
            else:
                tp_c, fp_c, fn_c, tn_c, classif_c, \
                tp_s, fp_s, fn_s, tn_s, classif_s = get_binary_metrics(gt_val_efetivo, pred_list, attr)

            df.at[idx, f'gt_raw_{attr}'] = valor_gt_bruto
            df.at[idx, f'gt_presente_{attr}'] = valor_presente_bruto
            df.at[idx, f'gt_efetivo_{attr}'] = gt_val_efetivo
            df.at[idx, f'pred_{attr}'] = str(pred_list) 
            df.at[idx, f'classif_contains_{attr}'] = classif_c
            df.at[idx, f'classif_strict_{attr}'] = classif_s

        loop_count = (idx - global_start) + 1
        if loop_count % SAVE_EVERY == 0 and loop_count > 0:
            try:
                # Salva TODAS as colunas (para o RESUME funcionar)
                df.to_csv(CAMINHO_CSV_PARCIAL, index=False, encoding='utf-8-sig')
                tqdm.write(f"--- Resultados parciais salvos em '{CAMINHO_CSV_PARCIAL}' (no índice {idx}) ---")
            except Exception as e:
                tqdm.write(f"--- ERRO ao salvar CSV parcial: {e} ---")

    # --- 5. Agregar métricas (ambos os tipos) ---
    print("\nProcessamento concluído. Agregando métricas da fatia processada...")
    tabela = {}
    df_processed_slice = df.iloc[global_start:global_end]

    for attr in AVALIAR.keys():
        tabela[attr] = {}
        if AVALIAR[attr]:
            classif_col_s = f'classif_strict_{attr}'; classif_col_c = f'classif_contains_{attr}'
            if classif_col_s not in df_processed_slice.columns:
                 tabela[attr]['contains'] = {'status': 'NÃO PROCESSADO'}
                 tabela[attr]['strict'] = {'status': 'NÃO PROCESSADO'}
                 continue
            
            valid_rows_s = df_processed_slice[classif_col_s].dropna()
            if len(valid_rows_s) == 0:
                tabela[attr]['strict'] = {'status': 'SEM LINHAS PROCESSADAS'}
            else:
                counts_s = valid_rows_s.value_counts()
                tp_s = counts_s.get('TP', 0); fp_s = counts_s.get('FP', 0)
                fn_s = counts_s.get('FN', 0); tn_s = counts_s.get('TN', 0)
                tabela[attr]['strict'] = calculate_prf1(tp_s, fp_s, fn_s, tn_s)

            valid_rows_c = df_processed_slice[classif_col_c].dropna()
            if len(valid_rows_c) == 0:
                 tabela[attr]['contains'] = {'status': 'SEM LINHAS PROCESSADAS'}
            else:
                counts_c = valid_rows_c.value_counts()
                tp_c = counts_c.get('TP', 0); fp_c = counts_c.get('FP', 0)
                fn_c = counts_c.get('FN', 0); tn_c = counts_c.get('TN', 0)
                tabela[attr]['contains'] = calculate_prf1(tp_c, fp_c, fn_c, tn_c)
        else:
            col_manual = COLUNAS_MANUAIS.get(attr)
            if col_manual:
                print(f"Calculando métricas manuais para '{attr}' da coluna '{col_manual}'...")
                try:
                    if col_manual not in df.columns:
                        tabela[attr]['contains'] = {'status': 'COLUNA MANUAL AUSENTE'}
                        tabela[attr]['strict'] = {'status': 'COLUNA MANUAL AUSENTE'}
                        continue
                    
                    mapeamento_classif = {'tp': 'tp', 'tp*': 'tp', 'fp': 'fp', 'fp*': 'fp', 'fn': 'fn', 'fn*': 'fn', 'tn': 'tn', 'tn*': 'tn', 'vn': 'tn', 'vn*': 'tn'}
                    clean_series = df[col_manual].astype(str).str.lower().str.strip().str.replace('*', '', regex=False)
                    counts = clean_series.map(mapeamento_classif).value_counts()
                    
                    tp_manual = counts.get('tp', 0); fp_manual = counts.get('fp', 0)
                    fn_manual = counts.get('fn', 0); tn_manual = counts.get('tn', 0)
                    
                    manual_metrics = calculate_prf1(tp_manual, fp_manual, fn_manual, tn_manual)
                    manual_metrics['status'] = 'MANUAL'
                    tabela[attr]['contains'] = manual_metrics.copy()
                    tabela[attr]['strict'] = manual_metrics.copy()
                except Exception as e:
                    print(f"ERRO ao calcular métricas manuais para '{attr}': {e}")
                    tabela[attr]['contains'] = {'status': 'ERRO MANUAL'}
                    tabela[attr]['strict'] = {'status': 'ERRO MANUAL'}
            else:
                tabela[attr]['contains'] = {'status': 'DESABILITADO'}
                tabela[attr]['strict'] = {'status': 'DESABILITADO'}

    # --- 6. Exportar CSV de análise (salvando o DF modificado) ---
    print(f"\nSalvando arquivo de análise final em '{CAMINHO_CSV_FINAL}'...")
    try:
        # Tentar reordenar colunas
        cols_order = ['texto_noticia']
        for attr in AVALIAR.keys():
             cols_order.extend(sorted([c for c in df.columns if c.endswith(f'_{attr}')]))
        original_cols = [c for c in df.columns if c not in cols_order]
        cols_order.extend(original_cols)
        
        # Filtrar colunas que realmente existem
        cols_order_existing = [c for c in cols_order if c in df.columns]
        df_export = df[cols_order_existing]
    except Exception:
        # Fallback se a reordenação falhar
        df_export = df
        
    try:
        df_export.to_csv(CAMINHO_CSV_FINAL, index=False, encoding='utf-8-sig')
        print(f"Arquivo de análise geral salvo com sucesso em: '{CAMINHO_CSV_FINAL}'")
    except Exception as e:
        print(f"ERRO ao salvar o CSV final: {e}")

    # --- 7. Imprimir Relatório Final (MODIFICADO) ---
    print("\n" + "="*80)
    print("                   RESULTADO DA AVALIAÇÃO (HÍBRIDO)")
    print(f"                   (Baseado na fatia {global_start} a {global_end-1})")
    print("="*80)

    print("\nNÍVEL 1: Métricas por Atributo (Métrica MENOS RESTRITIVA - 'Contém')")
    print_metrics_table(tabela, metric_type='contains')
    
    print("\nNÍVEL 1: Métricas por Atributo (Métrica RESTRITIVA - Acerto Exato)")
    print_metrics_table(tabela, metric_type='strict')
    
    total_noticias_avaliadas = len(df_processed_slice)
    
    # --- CÁLCULO NÍVEL 2 (MODIFICADO) ---
    all_accuracies_contains = []
    all_accuracies_strict = []
    attributes_in_level_2 = []
    
    for attr, metrics_by_type in tabela.items():
        metrics_c = metrics_by_type.get('contains')
        metrics_s = metrics_by_type.get('strict')
        
        if metrics_c and 'accuracy' in metrics_c: 
            all_accuracies_contains.append(metrics_c['accuracy'])
            attributes_in_level_2.append(attr) # Adiciona à lista
            
        if metrics_s and 'accuracy' in metrics_s:
            all_accuracies_strict.append(metrics_s['accuracy'])
            
    macro_avg_accuracy_total_contains = (sum(all_accuracies_contains) / len(all_accuracies_contains)) if all_accuracies_contains else 0.0
    macro_avg_accuracy_total_strict = (sum(all_accuracies_strict) / len(all_accuracies_strict)) if all_accuracies_strict else 0.0
    
    print("\n" + "="*80)
    print("NÍVEL 2: Métricas Gerais do Pipeline")
    print(f"(Baseado em TODOS os atributos: {', '.join(attributes_in_level_2)})")
    print("="*80)
    
    print(f"1. Média das Acurácias Individuais (baseada em acerto 'CONTÉM'):")
    print(f"   - Média das acurácias ('contém') dos {len(all_accuracies_contains)} atributos (auto+manual).")
    print(f"   - MÉDIA GERAL (Acurácia Média 'Contém'): {macro_avg_accuracy_total_contains:.2%}")
    print("\n")
    
    print(f"2. Média das Acurácias Individuais (baseada em acerto 'RESTRITO'):")
    print(f"   - Média das acurácias ('restritas') dos {len(all_accuracies_strict)} atributos (auto+manual).")
    print(f"   - MÉDIA GERAL (Acurácia Média 'Restrita'): {macro_avg_accuracy_total_strict:.2%}")
    
    print("="*80)
    print("\n")

# ==============================================================================
# EXECUTE THE SCRIPT
# ==============================================================================

if __name__ == "__main__":
    run_evaluation()

Iniciando pipeline de avaliação HÍBRIDO (Solr/Regex/LLM)...
Conectando ao Solr (http://localhost:8983/solr)...
Conexão ao Solr ('municipios') bem-sucedida.
Conectando ao Ollama (https://ollama-dev.ceos.ufsc.br) com modelo gpt-oss:20b...
Cliente Ollama preparado.
Carregando gabarito (Ground Truth) de '196_validation.csv'...
Total de 196 itens carregados. Processando fatia de 0 a 195 (196 itens).
Atributos a serem avaliados AUTOMATICAMENTE: municipio, modalidade, edital
Coluna marcador 'classif_strict_municipio' não encontrada. Desabilitando RESUME.


Avaliando notícias:  10%|█         | 20/196 [02:53<18:23,  6.27s/it] 

--- Resultados parciais salvos em 'analise_parcial_hibrido.csv' (no índice 19) ---


Avaliando notícias:  20%|██        | 40/196 [05:17<16:20,  6.28s/it]

--- Resultados parciais salvos em 'analise_parcial_hibrido.csv' (no índice 39) ---


...

--- Resultados parciais salvos em 'analise_parcial_hibrido.csv' (no índice 159) ---


Avaliando notícias:  92%|█████████▏| 180/196 [19:16<01:36,  6.01s/it]

--- Resultados parciais salvos em 'analise_parcial_hibrido.csv' (no índice 179) ---


Avaliando notícias: 100%|██████████| 196/196 [34:05<00:00, 10.44s/it] 


Processamento concluído. Agregando métricas da fatia processada...
Calculando métricas manuais para 'objeto' da coluna 'validaçao_objeto'...

Salvando arquivo de análise final em 'analise_geral_hibrido.csv'...
Arquivo de análise geral salvo com sucesso em: 'analise_geral_hibrido.csv'

                   RESULTADO DA AVALIAÇÃO (HÍBRIDO)
                   (Baseado na fatia 0 a 195)

NÍVEL 1: Métricas por Atributo (Métrica MENOS RESTRITIVA - 'Contém')
-------------------------------------------------------------------------------------
| Atributo          | Acurácia  | Precisão  | Recall    | F1-Score  | TP   | FP   | FN   | TN   |
-------------------------------------------------------------------------------------
| municipio         | 66.84%    | 84.92%    | 69.93%    | 76.70%    | 107  | 19   | 46   | 24   |
| modalidade        | 89.29%    | 81.82%    | 80.36%    | 81.08%    | 45   | 10   | 11   | 130  |
| edital            | 95.41%    | 62.50%    | 100.00%   | 76.92%    | 15   | 9 